# Aircraft Maintenance Data Pipeline and Feature Engineering

Files for the team:
- `labels.csv` for everyone
- `tabular_features.csv` for See Lin (XGBoost, unscaled)
- `tabular_features_scaled.csv` scaled version (optional)
- `sequences.npy` for Tanya / Linh / Helen (TS models, scaled float16)
- `seq_indices.npy`  Master Index alignment for sequences
- `channel_scaler.pkl` + `tabular_scaler.pkl`  for inference

NEW COLUMNS REFERENCE

labels.csv
rul_2d            — PRIMARY TARGET (binary)
                      0 = at risk: before_after=="before" AND date_diff >= -2
                          (flight within 2 days before maintenance)
                      1 = safe:    before_after=="after"
                          OR (before_after=="before" AND date_diff < -2)
                          (post-maintenance OR more than 2 days before)
            

rul_5d  and rul_10d ->same logic as 2d, to use if we expand our models

label_grouped     — top-10 maintenance labels kept as-is,
                      all remaining rare labels collapsed to "other"
                      (stretch goal: 11-class classification)

split             — train / val / test
                      70% train / 15% val / 15% test
                      stratified on rul_2d to preserve class balance
                      across all three splits

tabular_features.csv

All sensor feature columns follow naming: {channel}__{stat}

KEEP_CHANNELS -> redundant channels dropped based on EDA
(25 pairs with |r| > 0.9 → kept one representative per group):
  volt1             OAT
  amp1              IAS
  FQtyL             VSpd
  FQtyR             NormAc
  E1 FFlow          AltMSL
  E1 OilT
  E1 OilP
  E1 RPM
  E1 CHT1           (CHT2/3/4 dropped: r > 0.983 with CHT1)
  E1 EGT1           (EGT2/3/4 dropped: r > 0.949 with EGT1)

DROPPED CHANNELS:
  volt2             (r = 0.937 with volt1)
  amp2              (r > 0.9 with amp1, high missingness)
  E1 CHT2/3/4       (r > 0.983 with E1 CHT1)
  E1 EGT2/3/4       (r > 0.949 with E1 EGT1)

Per-channel stats :
  {ch}__mean        — arithmetic mean across all timesteps
  {ch}__miss_flag   — 1 if channel was entirely missing for this flight
                      (filled to 0 by imputation pipeline), 0 otherwise
etc...

Flight phase proportions:
  phase__ground_pct   — fraction of timesteps with E1 RPM <= 1000
                        (engine at ground/taxi RPM)
  phase__climb_pct    — fraction with RPM > 1000 AND VSpd > +100 fpm
  phase__cruise_pct   — fraction with RPM > 1000 AND -100 <= VSpd <= +100 fpm
  phase__descent_pct  — fraction with RPM > 1000 AND VSpd < -100 fpm

Cross-channel physics features:
  cross__fuel_efficiency  — mean(E1 FFlow / E1 RPM) where RPM > 800
                            fuel consumed per revolution
                            high values may indicate rich mixture or
                            dirty fuel injectors
  cross__oil_stress       — mean(E1 OilT / E1 OilP) where OilP > 10
                            oil temperature-to-pressure ratio
                            high values indicate stressed lubrication
  cross__cht_egt_ratio    — mean(E1 CHT1 / E1 EGT1) where EGT1 > 100
                            cylinder head to exhaust gas temp ratio
                            proxy for combustion efficiency
  cross__total_fuel       — mean(FQtyL + FQtyR) across flight
                            mean total fuel quantity
  cross__fuel_imbalance   — mean(|FQtyL - FQtyR|) across flight
                            asymmetric fuel burn between tanks
  cross__volt_drop        — mean(volt1) - min(volt1)
                            voltage drop range across flight
                            high values may indicate alternator or
                            battery degradation
  cross__elec_load        — mean(amp1)
                            mean electrical current draw

Flight metadata:
  meta__flight_length     — number of timesteps in this flight
                            (same as flight_length in labels.csv)


sequences.npy

Shape: (N_flights, 9212, 15)  dtype: float16

Axis 0 — flights (aligned to seq_indices.npy)
Axis 1 — timesteps
          fixed length = 9212 (p95 from EDA)
          flights longer than 9212: last 9212 steps kept (most recent)
          flights shorter than 9212: left-padded with 0
Axis 2 — channels (in this exact order):
          0: volt1      1: amp1       2: FQtyL      3: FQtyR
          4: E1 FFlow   5: E1 OilT    6: E1 OilP   7: E1 RPM
          8: E1 CHT1    9: E1 EGT1   10: OAT      11: IAS
          12: VSpd      13: NormAc   14: AltMSL

Preprocessing applied:
  - NaN imputed per flight (ffill → bfill → 0 for full-channel missing)
  - RobustScaler fitted on training flights only (no leakage)
  - Encoded as float16 to reduce file size (~6 GB vs ~12 GB in float32)

seq_indices.npy

Shape: (N_flights,)  dtype: int64
Master Index value for each row in sequences.npy
Use to align sequences with labels.csv:
  y     = labels.loc[seq_indices, 'rul_2d'].values
  split = labels.loc[seq_indices, 'split'].values
  

In [1]:
#Imports & Configuration

import os, gc, warnings, pickle
import numpy as np
import pandas as pd
import dask.dataframe as dd
from numpy.lib.format import open_memmap
from scipy.stats import skew as sp_skew, kurtosis as sp_kurtosis
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

# Paths
FULL_DATA_DIR = '/kaggle/input/datasets/hooong/aviation-maintenance-dataset-from-the-ngafid/all_flights/all_flights'
OUT_DIR       = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

ALL_CHANNELS = [
    'volt1','volt2','amp1','amp2','FQtyL','FQtyR',
    'E1 FFlow','E1 OilT','E1 OilP','E1 RPM',
    'E1 CHT1','E1 CHT2','E1 CHT3','E1 CHT4',
    'E1 EGT1','E1 EGT2','E1 EGT3','E1 EGT4',
    'OAT','IAS','VSpd','NormAc','AltMSL'
]

# Drop redundant channels
# Keep one representative per correlated group
DROP_CHANNELS = ['volt2','amp2','E1 CHT2','E1 CHT3','E1 CHT4','E1 EGT2','E1 EGT3','E1 EGT4']
KEEP_CHANNELS = [c for c in ALL_CHANNELS if c not in DROP_CHANNELS]
# Result: 15 channels

# Pipeline settings
SEQ_LEN      = 9212  
SHORT_CUTOFF = 100  
TAU_VALUES   = [2, 5, 10] 
RANDOM_SEED  = 42

print(f'Keep channels ({len(KEEP_CHANNELS)}): {KEEP_CHANNELS}')
print(f'Sequence length (p95): {SEQ_LEN}')
print(f'RUL thresholds (days): {TAU_VALUES}')

Keep channels (15): ['volt1', 'amp1', 'FQtyL', 'FQtyR', 'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM', 'E1 CHT1', 'E1 EGT1', 'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL']
Sequence length (p95): 9212
RUL thresholds (days): [2, 5, 10]


In [2]:
# Load header, filter, derive labels, split
flight_header_df = pd.read_csv(
    os.path.join(FULL_DATA_DIR, 'flight_header.csv'), index_col='Master Index'
)
flight_data_df = dd.read_parquet(os.path.join(FULL_DATA_DIR, 'one_parq'))
print(f'Raw flights       : {len(flight_header_df):,}')
print(f'Parquet partitions: {flight_data_df.npartitions}')
#Remove short flights
valid_header = flight_header_df[flight_header_df['flight_length'] >= SHORT_CUTOFF].copy()
print(f'\nAfter short-flight filter : {len(valid_header):,}  '
      f'(removed {len(flight_header_df)-len(valid_header):,} flights, '
      f'{(1-len(valid_header)/len(flight_header_df))*100:.1f}%)')
#Exclude maintenance-day flights
labeled = valid_header[valid_header['before_after'] != 'same'].copy()
print(f'After excluding same     : {len(labeled):,}  '
      f'(removed {len(valid_header)-len(labeled):,} flights)')
# Step 3: Derive binary RUL targets at each tau
# rul = 1 (SAFE): flight is after maintenance OR before but > t days away
# rul = 0 (RISK): flight is before AND within t days of maintenance
for tau in TAU_VALUES:
    labeled[f'rul_{tau}d'] = (
        (labeled['before_after'] == 'after') |
        (labeled['date_diff'] < -tau)
    ).astype(int)
print('\nRUL class balance:')
for tau in TAU_VALUES:
    vc = labeled[f'rul_{tau}d'].value_counts().sort_index()
    n  = len(labeled)
    print(f'  tau={tau:2d}d -> risk(0): {vc.get(0,0):,} ({vc.get(0,0)/n*100:.1f}%)  '
          f'safe(1): {vc.get(1,0):,} ({vc.get(1,0)/n*100:.1f}%)')
#Top-10 label grouping for stretch goal
top10 = labeled['label'].value_counts().nlargest(10).index
labeled['label_grouped'] = labeled['label'].where(labeled['label'].isin(top10), other='other')
print(f'\nTop-10 labels cover {labeled["label"].isin(top10).mean()*100:.1f}% of flights')
#Stratified 70/15/15 split on rul_2d
train_idx, temp_idx = train_test_split(
    labeled.index, test_size=0.30,
    stratify=labeled['rul_2d'], random_state=RANDOM_SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=labeled.loc[temp_idx, 'rul_2d'], random_state=RANDOM_SEED
)
labeled['split'] = 'train'
labeled.loc[val_idx,  'split'] = 'val'
labeled.loc[test_idx, 'split'] = 'test'
print('\nSplit summary:')
for sp in ['train','val','test']:
    sub = labeled[labeled['split']==sp]
    vc  = sub['rul_2d'].value_counts(normalize=True)
    print(f'  {sp:5s}: {len(sub):,} | class0: {vc.get(0,0)*100:.1f}%  class1: {vc.get(1,0)*100:.1f}%')
labeled[['before_after','date_diff','flight_length','label',
         'label_grouped','rul_2d','rul_5d','rul_10d','split',
         'number_flights_before']].to_csv(f'{OUT_DIR}/labels.csv')
print(f'\nSaved: labels.csv  ({len(labeled):,} flights)')

Raw flights       : 28,935
Parquet partitions: 401

After short-flight filter : 24,934  (removed 4,001 flights, 13.8%)
After excluding same     : 19,540  (removed 5,394 flights)

RUL class balance:
  tau= 2d -> risk(0): 7,889 (40.4%)  safe(1): 11,651 (59.6%)
  tau= 5d -> risk(0): 10,119 (51.8%)  safe(1): 9,421 (48.2%)
  tau=10d -> risk(0): 10,464 (53.6%)  safe(1): 9,076 (46.4%)

Top-10 labels cover 79.3% of flights

Split summary:
  train: 13,678 | class0: 40.4%  class1: 59.6%
  val  : 2,931 | class0: 40.4%  class1: 59.6%
  test : 2,931 | class0: 40.4%  class1: 59.6%

Saved: labels.csv  (19,540 flights)


In [3]:
flight_header_df.head()

,before_after,date_diff,flight_length,label,hierarchy,number_flights_before
Master Index,,,,,,
1,before,-1,4723.0,intake gasket leak/damage,NaN,0
2,before,-2,4649.0,intake gasket leak/damage,NaN,3
3,same,0,40.0,intake gasket leak/damage,NaN,-1
4,before,0,14.0,intake gasket leak/damage,NaN,0
5,same,0,683.0,intake gasket leak/damage,NaN,-1


In [4]:
pd.read_csv('/kaggle/working/labels.csv').head()

,Master Index,before_after,date_diff,flight_length,label,label_grouped,rul_2d,rul_5d,rul_10d,split,number_flights_before
0,1,before,-1,4723.0,intake gasket leak/damage,intake gasket leak/damage,0,0,0,train,0
1,2,before,-2,4649.0,intake gasket leak/damage,intake gasket leak/damage,0,0,0,val,3
2,7,after,1,3482.0,intake gasket leak/damage,intake gasket leak/damage,1,1,1,train,-1
3,9,before,-1,4979.0,intake gasket leak/damage,intake gasket leak/damage,0,0,0,test,0
4,11,after,2,5204.0,intake gasket leak/damage,intake gasket leak/damage,1,1,1,train,-1


In [5]:
sample_flight = flight_data_df.get_partition(0).compute()
display(sample_flight.head(10))

,volt1,volt2,amp1,amp2,FQtyL,FQtyR,E1 FFlow,E1 OilT,E1 OilP,E1 RPM,...,E1 EGT2,E1 EGT3,E1 EGT4,OAT,IAS,VSpd,NormAc,AltMSL,timestep,cluster
Master Index,,,,,,,,,,,,,,,,,,,,,
1,28.8,NaN,0.8,NaN,48.89,44.06,13.28,168.55,82.51,2519.7,...,1317.44,1298.97,1322.83,12.8,144.89,29.61,0.01,3010.7,3151,c_28
1,28.8,NaN,0.8,NaN,48.93,44.06,13.31,168.60,82.51,2519.4,...,1316.64,1300.68,1322.15,12.8,144.53,49.78,-0.00,3011.3,3150,c_28
1,28.8,NaN,0.9,NaN,48.96,44.06,13.36,168.65,82.51,2520.3,...,1317.20,1301.05,1323.12,12.8,144.15,58.68,-0.01,3011.8,3149,c_28
1,28.8,NaN,0.7,NaN,48.86,44.06,13.30,168.64,82.51,2519.3,...,1317.73,1302.14,1321.93,12.8,143.79,47.31,-0.00,3012.2,3148,c_28
1,28.8,NaN,0.6,NaN,48.87,44.06,13.30,168.65,82.51,2518.8,...,1318.00,1308.52,1316.99,12.8,143.45,36.05,-0.01,3012.3,3147,c_28
1,28.8,NaN,0.8,NaN,48.78,44.07,13.27,168.78,82.50,2518.3,...,1326.33,1304.53,1319.91,12.8,142.05,6.07,0.00,3012.3,3143,c_28
1,28.8,NaN,1.0,NaN,48.92,44.06,13.38,168.62,82.51,2518.4,...,1320.08,1306.83,1316.62,12.8,142.77,42.54,0.01,3012.5,3145,c_28
1,28.8,NaN,1.0,NaN,48.85,44.06,13.30,168.71,82.51,2508.8,...,1321.58,1304.24,1320.08,12.8,142.42,27.62,0.00,3012.4,3144,c_28
1,28.8,NaN,0.8,NaN,48.84,44.07,13.26,168.76,82.51,2519.9,...,1327.16,1304.34,1319.09,12.8,141.66,-23.88,0.00,3012.3,3142,c_28


In [6]:
# Summary stats for metadata and label columns
print("=" * 60)
print("METADATA + LABEL SUMMARY STATISTICS")
print("=" * 60)

# Numeric columns — full describe
numeric_cols = ['date_diff', 'flight_length', 'number_flights_before',
                'rul_2d', 'rul_5d', 'rul_10d']
print("\nNumeric columns:")
display(labeled[numeric_cols].describe())

# Master Index (it's the index, so describe separately)
print("\nMaster Index range:")
print(f"  min: {labeled.index.min()}")
print(f"  max: {labeled.index.max()}")
print(f"  count: {labeled.index.nunique():,} unique flights")

# Categorical columns — value counts
print("\nbefore_after distribution:")
print(labeled['before_after'].value_counts())

print("\nrul_2d distribution (primary target):")
print(labeled['rul_2d'].value_counts())
print(labeled['rul_2d'].value_counts(normalize=True).round(3))

print("\nTop 10 labels:")
print(labeled['label'].value_counts().head(10))

METADATA + LABEL SUMMARY STATISTICS

Numeric columns:


,date_diff,flight_length,number_flights_before,rul_2d,rul_5d,rul_10d
count,19540.000000,19540.000000,19540.000000,19540.000000,19540.000000,19540.000000
mean,-0.076510,5058.655629,0.455067,0.596264,0.482139,0.464483
std,4.082361,2582.831206,1.728835,0.490658,0.499694,0.498750
min,-108.000000,107.000000,-1.000000,0.000000,0.000000,0.000000
25%,-2.000000,3812.000000,-1.000000,0.000000,0.000000,0.000000
50%,0.000000,5121.000000,0.000000,1.000000,0.000000,0.000000
75%,1.000000,6132.250000,2.000000,1.000000,1.000000,1.000000
max,70.000000,30059.000000,4.000000,1.000000,1.000000,1.000000



Master Index range:
  min: 1
  max: 32820
  count: 19,540 unique flights

before_after distribution:
before_after
before    10591
after      8949
Name: count, dtype: int64

rul_2d distribution (primary target):
rul_2d
1    11651
0     7889
Name: count, dtype: int64
rul_2d
1    0.596
0    0.404
Name: proportion, dtype: float64

Top 10 labels:
label
intake gasket leak/damage                     6814
rocker cover leak/loose/damage                3575
intake tube/bolt/seal/boot loose or damage     898
baffle crack/damage/loose/miss                 890
baffle plug need repair/replace                776
baffle screw miss/loose                        599
baffle seal loose/damage                       577
engine run rough                               473
baffle tie/tie rod loose or damage             448
cylinder compression issue                     439
Name: count, dtype: int64


In [7]:
print("ALL 23 SENSORS:")
for i, ch in enumerate(ALL_CHANNELS, 1):
    print(f"  {i:2d}. {ch}")

print(f"\n15 KEPT after dropping redundant channels:")
for ch in KEEP_CHANNELS:
    print(f"  - {ch}")

print(f"\n8 DROPPED (|r| > 0.9 redundant):")
for ch in DROP_CHANNELS:
    print(f"  - {ch}")

ALL 23 SENSORS:
   1. volt1
   2. volt2
   3. amp1
   4. amp2
   5. FQtyL
   6. FQtyR
   7. E1 FFlow
   8. E1 OilT
   9. E1 OilP
  10. E1 RPM
  11. E1 CHT1
  12. E1 CHT2
  13. E1 CHT3
  14. E1 CHT4
  15. E1 EGT1
  16. E1 EGT2
  17. E1 EGT3
  18. E1 EGT4
  19. OAT
  20. IAS
  21. VSpd
  22. NormAc
  23. AltMSL

15 KEPT after dropping redundant channels:
  - volt1
  - amp1
  - FQtyL
  - FQtyR
  - E1 FFlow
  - E1 OilT
  - E1 OilP
  - E1 RPM
  - E1 CHT1
  - E1 EGT1
  - OAT
  - IAS
  - VSpd
  - NormAc
  - AltMSL

8 DROPPED (|r| > 0.9 redundant):
  - volt2
  - amp2
  - E1 CHT2
  - E1 CHT3
  - E1 CHT4
  - E1 EGT2
  - E1 EGT3
  - E1 EGT4


In [8]:
# SPLIT VISUALIZATIONS
import matplotlib.pyplot as plt
import numpy as np

RISK = '#D7263D'   # class 0 — at risk
SAFE = '#1B998B'   # class 1 — safe
DARK = '#22223B'
GREY = '#6c757d'

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

#Pull REAL numbers from the current stratified split
sr = {}
for sp in ['train', 'val', 'test']:
    sub = labeled[labeled['split'] == sp]
    n   = len(sub)
    c0  = sub['rul_2d'].value_counts(normalize=True).get(0, 0) * 100
    sr[sp] = (n, c0, 100 - c0)

overall_c0 = labeled['rul_2d'].value_counts(normalize=True).get(0, 0) * 100
overall_n0 = (labeled['rul_2d'] == 0).sum()
overall_n1 = (labeled['rul_2d'] == 1).sum()
total_n    = len(labeled)

#Grouped-split numbers across seeds
seeds    = ['seed=42', 'seed=0', 'seed=123', 'seed=7']
train_c0 = [40.9, 56.0, 43.6, 45.5]
val_c0   = [45.2, 22.7, 25.8, 22.6]
test_c0  = [35.1, 29.7, 58.3, 57.1]

#Stratified random: stacked balance
fig, ax = plt.subplots(figsize=(8, 5))
splits = [f'Train\n({sr["train"][0]:,})', f'Val\n({sr["val"][0]:,})', f'Test\n({sr["test"][0]:,})']
c0 = [sr['train'][1], sr['val'][1], sr['test'][1]]
c1 = [sr['train'][2], sr['val'][2], sr['test'][2]]
x = np.arange(len(splits))
ax.bar(x, c0, 0.55, label='Class 0 — At risk', color=RISK)
ax.bar(x, c1, 0.55, bottom=c0, label='Class 1 — Safe', color=SAFE)
for i in range(3):
    ax.text(i, c0[i]/2, f'{c0[i]:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=12)
    ax.text(i, c0[i]+c1[i]/2, f'{c1[i]:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel('Class proportion (%)'); ax.set_ylim(0, 100)
ax.set_title('Stratified Random Split — Identical Balance Across All Splits', fontweight='bold', pad=15)
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout(); plt.savefig('/kaggle/working/fig1_stratified_random.png', dpi=200, bbox_inches='tight'); plt.close()
print("Saved fig1_stratified_random.png")


#Grouped split: unstable across seeds
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(seeds)); w = 0.25
ax.bar(x - w, train_c0, w, label='Train', color='#264653')
ax.bar(x,     val_c0,   w, label='Val',   color='#E9C46A')
ax.bar(x + w, test_c0,  w, label='Test',  color='#E76F51')
ax.axhline(overall_c0, color=RISK, ls='--', lw=1.5, alpha=0.7)
ax.text(len(seeds)-0.7, overall_c0+1.5, f'Ideal ({overall_c0:.1f}%)', fontsize=8, color=RISK)
ax.set_xticks(x); ax.set_xticklabels(seeds)
ax.set_ylabel('Class 0 proportion (%)'); ax.set_ylim(0, 70)
ax.set_title('Event-Grouped Split — Class Balance Swings by Seed', fontweight='bold', pad=15)
ax.legend(loc='upper left', framealpha=0.9, ncol=3)
plt.tight_layout(); plt.savefig('/kaggle/working/fig2_grouped_unstable.png', dpi=200, bbox_inches='tight'); plt.close()
print("Saved fig2_grouped_unstable.png")

#Side-by-side decision summary (the key slide image)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.bar(['Train','Val','Test'], c0, color=RISK, width=0.5, alpha=0.85)
ax1.set_ylim(0, 70); ax1.set_ylabel('Class 0 proportion (%)')
ax1.set_title('Stratified Random\nStable  /  Some leakage', fontweight='bold', fontsize=12)
for i, v in enumerate(c0):
    ax1.text(i, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
ax2.bar(['s=42','s=0','s=123','s=7'], test_c0, color='#E76F51', width=0.5, alpha=0.85)
ax2.axhline(overall_c0, color=RISK, ls='--', lw=1.5, alpha=0.7)
ax2.text(3.4, overall_c0+1.5, 'Ideal', fontsize=8, color=RISK, ha='center')
ax2.set_ylim(0, 70); ax2.set_ylabel('Test Class 0 proportion (%)')
ax2.set_title('Event-Grouped (StratifiedGroupKFold)\nZero leakage  /  Unstable', fontweight='bold', fontsize=12)
for i, v in enumerate(test_c0):
    ax2.text(i, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
fig.suptitle('Split Strategy Comparison — Why We Chose Stratified Random', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('/kaggle/working/fig3_decision_summary.png', dpi=200, bbox_inches='tight'); plt.close()
print("Saved fig3_decision_summary.png")

#Overall class balance pie
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    [overall_c0, 100-overall_c0],
    labels=[f'At risk\n({overall_n0:,} flights)', f'Safe\n({overall_n1:,} flights)'],
    colors=[RISK, SAFE], autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 11, 'fontweight':'bold'},
    wedgeprops={'edgecolor':'white','linewidth':2}
)
for at in autotexts: at.set_color('white'); at.set_fontsize(13)
ax.set_title(f'RUL Target Distribution (tau = 2 days)\n{total_n:,} flights total', fontweight='bold', pad=15)
plt.tight_layout(); plt.savefig('/kaggle/working/fig4_class_balance.png', dpi=200, bbox_inches='tight'); plt.close()
print("Saved fig4_class_balance.png")

print("\nAll 4 figures saved to /kaggle/working/ — download from the Output tab.")

Saved fig1_stratified_random.png
Saved fig2_grouped_unstable.png
Saved fig3_decision_summary.png
Saved fig4_class_balance.png

All 4 figures saved to /kaggle/working/ — download from the Output tab.


In [9]:
# SENSOR DISTRIBUTION ANALYSES for methodology / EDA slides
#Generates 3 figures that statistically justify our labelling and aggregation assumptions:
#fig5 T-1 vs T+1  (does maintenance fix a real problem?)
#fig6 operating-threshold consistency across the dataset
#fig7  T+1 vs T+2  (do post maintenance aircraft share a baseline?)
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp

RISK = '#D7263D'
SAFE = '#1B998B'
BLUE = '#264653'
GOLD = '#E9C46A'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

KEY_CHANNELS = ['E1 RPM', 'E1 CHT1', 'E1 EGT1', 'E1 OilT', 'E1 OilP', 'E1 FFlow']

#sample raw sensor rows for a set of flight IDs
def sample_flights(flight_ids, n=200, seed=42):
    rng = np.random.default_rng(seed)
    if len(flight_ids) > n:
        flight_ids = rng.choice(np.array(flight_ids), size=n, replace=False)
    flight_ids = set(flight_ids)
    parts = []
    for part_i in range(flight_data_df.npartitions):
        part = flight_data_df.get_partition(part_i).compute()
        part = part[part.index.isin(flight_ids)]
        if len(part) > 0:
            parts.append(part)
        del part
    return pd.concat(parts) if parts else pd.DataFrame()

#Define flight cohorts by date_diff
t_minus1 = flight_header_df[flight_header_df['date_diff'] == -1].index.tolist()
t_plus1  = flight_header_df[flight_header_df['date_diff'] ==  1].index.tolist()
t_plus2  = flight_header_df[flight_header_df['date_diff'] ==  2].index.tolist()

print(f"T-1 flights: {len(t_minus1):,}")
print(f"T+1 flights: {len(t_plus1):,}")
print(f"T+2 flights: {len(t_plus2):,}")

print("\nLoading sensor samples (this takes a few minutes)...")
df_tm1 = sample_flights(t_minus1)
df_tp1 = sample_flights(t_plus1)
df_tp2 = sample_flights(t_plus2)
print("Done loading.")


#FIG 5 
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
ks_results = {}

for i, ch in enumerate(KEY_CHANNELS):
    ax = axes[i]
    if ch in df_tm1.columns and ch in df_tp1.columns:
        v1  = df_tm1[ch].dropna()
        vp1 = df_tp1[ch].dropna()
        lo  = min(v1.quantile(0.01), vp1.quantile(0.01))
        hi  = max(v1.quantile(0.99), vp1.quantile(0.99))
        bins = np.linspace(lo, hi, 50)
        ax.hist(v1,  bins=bins, alpha=0.6, color=RISK, density=True, label='T-1 (before)')
        ax.hist(vp1, bins=bins, alpha=0.6, color=SAFE, density=True, label='T+1 (after)')
        # KS test for distribution difference
        ks_stat, ks_p = ks_2samp(v1, vp1)
        ks_results[ch] = (ks_stat, ks_p)
        sig = '***' if ks_p < 0.001 else ('**' if ks_p < 0.01 else ('*' if ks_p < 0.05 else 'ns'))
        ax.set_title(f'{ch}  (KS={ks_stat:.3f} {sig})', fontweight='bold', fontsize=10)
        ax.set_xlabel('Sensor value'); ax.set_ylabel('Density')
        if i == 0:
            ax.legend(fontsize=9)

fig.suptitle('Sensor Distributions: 1 Day Before (T-1) vs 1 Day After (T+1) Maintenance\n'
             'Significant shifts (KS test) support that maintenance addresses real faults',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/fig5_t1_vs_tp1.png', dpi=200, bbox_inches='tight')
plt.close()
print("\nSaved fig5_t1_vs_tp1.png")
print("KS test results (T-1 vs T+1):")
for ch, (stat, p) in ks_results.items():
    print(f"  {ch:12s}: KS={stat:.3f}  p={p:.2e}")


# FIG 6: Operating-threshold consistency across dataset
# Shows overall sensor ranges with normal operating bands

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# Use a broad random sample across all flights
all_sample_ids = flight_header_df.sample(n=min(400, len(flight_header_df)), random_state=42).index.tolist()
df_all = sample_flights(all_sample_ids, n=400)

for i, ch in enumerate(KEY_CHANNELS):
    ax = axes[i]
    if ch in df_all.columns:
        vals = df_all[ch].dropna()
        vals = vals[(vals > vals.quantile(0.01)) & (vals < vals.quantile(0.99))]
        ax.hist(vals, bins=60, color=BLUE, alpha=0.75, density=True)
        # Mark the normal operating band (p25-p75)
        p25, p75 = vals.quantile(0.25), vals.quantile(0.75)
        ax.axvspan(p25, p75, alpha=0.2, color=SAFE, label='Normal band (IQR)')
        ax.axvline(vals.median(), color=RISK, ls='--', lw=1.5, label='Median')
        ax.set_title(ch, fontweight='bold', fontsize=10)
        ax.set_xlabel('Sensor value'); ax.set_ylabel('Density')
        if i == 0:
            ax.legend(fontsize=8)

fig.suptitle('Normal Operating Ranges Across All Flights\n'
             'Consistent sensor bands confirm uniform operating standards across the fleet',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/fig6_operating_ranges.png', dpi=200, bbox_inches='tight')
plt.close()
print("Saved fig6_operating_ranges.png")

# FIG 7 

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
overlap_results = {}

for i, ch in enumerate(KEY_CHANNELS):
    ax = axes[i]
    if ch in df_tp1.columns and ch in df_tp2.columns:
        vp1 = df_tp1[ch].dropna()
        vp2 = df_tp2[ch].dropna()
        lo  = min(vp1.quantile(0.01), vp2.quantile(0.01))
        hi  = max(vp1.quantile(0.99), vp2.quantile(0.99))
        bins = np.linspace(lo, hi, 50)
        ax.hist(vp1, bins=bins, alpha=0.6, color=SAFE, density=True, label='T+1')
        ax.hist(vp2, bins=bins, alpha=0.6, color=GOLD, density=True, label='T+2')
        ks_stat, ks_p = ks_2samp(vp1, vp2)
        overlap_results[ch] = (ks_stat, ks_p)
        # For overlap we WANT non-significant (distributions same)
        verdict = 'overlap' if ks_p > 0.05 else 'differ'
        ax.set_title(f'{ch}  (KS={ks_stat:.3f}, {verdict})', fontweight='bold', fontsize=10)
        ax.set_xlabel('Sensor value'); ax.set_ylabel('Density')
        if i == 0:
            ax.legend(fontsize=9)

fig.suptitle('Sensor Distributions: T+1 vs T+2 (Both Post-Maintenance)\n'
             'Tight overlap confirms aircraft share a common baseline, justifying model aggregation',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/fig7_tp1_vs_tp2.png', dpi=200, bbox_inches='tight')
plt.close()
print("Saved fig7_tp1_vs_tp2.png")
print("KS test results (T+1 vs T+2):")
for ch, (stat, p) in overlap_results.items():
    print(f"  {ch:12s}: KS={stat:.3f}  p={p:.2e}  ({'overlap' if p>0.05 else 'differ'})")

print("\nAll 3 sensor figures saved to /kaggle/working/ — download from Output tab.")

T-1 flights: 4,400
T+1 flights: 5,606
T+2 flights: 2,039

Loading sensor samples (this takes a few minutes)...
Done loading.

Saved fig5_t1_vs_tp1.png
KS test results (T-1 vs T+1):
  E1 RPM      : KS=0.080  p=0.00e+00
  E1 CHT1     : KS=0.039  p=0.00e+00
  E1 EGT1     : KS=0.069  p=0.00e+00
  E1 OilT     : KS=0.055  p=0.00e+00
  E1 OilP     : KS=0.128  p=0.00e+00
  E1 FFlow    : KS=0.075  p=0.00e+00
Saved fig6_operating_ranges.png
Saved fig7_tp1_vs_tp2.png
KS test results (T+1 vs T+2):
  E1 RPM      : KS=0.037  p=0.00e+00  (differ)
  E1 CHT1     : KS=0.024  p=6.38e-231  (differ)
  E1 EGT1     : KS=0.031  p=0.00e+00  (differ)
  E1 OilT     : KS=0.090  p=0.00e+00  (differ)
  E1 OilP     : KS=0.043  p=0.00e+00  (differ)
  E1 FFlow    : KS=0.029  p=0.00e+00  (differ)

All 3 sensor figures saved to /kaggle/working/ — download from Output tab.


In [10]:
flight_header_df.head()

,before_after,date_diff,flight_length,label,hierarchy,number_flights_before
Master Index,,,,,,
1,before,-1,4723.0,intake gasket leak/damage,NaN,0
2,before,-2,4649.0,intake gasket leak/damage,NaN,3
3,same,0,40.0,intake gasket leak/damage,NaN,-1
4,before,0,14.0,intake gasket leak/damage,NaN,0
5,same,0,683.0,intake gasket leak/damage,NaN,-1


In [11]:
# EDA SLIDE FIGURES
import matplotlib.pyplot as plt
import numpy as np

RISK = '#D7263D'
SAFE = '#1B998B'
BLUE = '#264653'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})


# FIG 8 Flight length distribution (justifies SEQ_LEN=9212)

fig, ax = plt.subplots(figsize=(9, 5))

fl = flight_header_df['flight_length'].dropna()
# Clip extreme tail for readability (max is 30,059 but rare)
fl_clipped = fl[fl <= 15000]

ax.hist(fl_clipped, bins=80, color=BLUE, alpha=0.8, edgecolor='white', linewidth=0.3)

# Mark key statistics
p95 = fl.quantile(0.95)
median = fl.median()
ax.axvline(median, color=SAFE, ls='--', lw=2, label=f'Median = {median:,.0f}')
ax.axvline(p95, color=RISK, ls='--', lw=2, label=f'p95 = {p95:,.0f}  (SEQ_LEN)')
ax.axvline(100, color='grey', ls=':', lw=1.5, label='Cutoff = 100 (short flights removed)')

ax.set_xlabel('Flight length (timesteps)')
ax.set_ylabel('Number of flights')
ax.set_title('Flight Length Distribution\nLong right tail justifies truncation at p95 = 9,212 timesteps',
             fontweight='bold', pad=15)
ax.legend(loc='upper right', framealpha=0.95)

# Annotate the tail
ax.annotate(f'Max = {fl.max():,.0f}\n(clipped from view)',
            xy=(14500, ax.get_ylim()[1]*0.3), fontsize=8, color='grey',
            ha='right', style='italic')

plt.tight_layout()
plt.savefig('/kaggle/working/fig8_flight_length.png', dpi=200, bbox_inches='tight')
plt.close()
print("Saved fig8_flight_length.png")
print(f"  Flight length — median: {median:,.0f}, p95: {p95:,.0f}, max: {fl.max():,.0f}")


# FIG 9 Correlation heatmap (justifies dropping 8 channels)
print("\nSampling sensor data for correlation heatmap...")
sample_ids = flight_header_df.sample(n=min(300, len(flight_header_df)), random_state=42).index
sample_ids = set(sample_ids)

corr_rows = []
for part_i in range(flight_data_df.npartitions):
    part = flight_data_df.get_partition(part_i).compute()
    part = part[part.index.isin(sample_ids)]
    if len(part) > 0:
        sensor_cols = [c for c in ALL_CHANNELS if c in part.columns]
        corr_rows.append(part[sensor_cols])
    del part
    # Stop once we have enough rows for a stable estimate
    if sum(len(r) for r in corr_rows) > 200000:
        break

corr_data = pd.concat(corr_rows)
print(f"  Computed correlation on {len(corr_data):,} sensor rows")

corr = corr_data.corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)

# Annotate cells with |r| > 0.9 (the ones that drove channel dropping)
high_corr_count = 0
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        val = corr.iloc[i, j]
        if i != j and abs(val) > 0.9:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    color='white' if abs(val) > 0.95 else 'black')
            high_corr_count += 1

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Pearson correlation', fontsize=10)

ax.set_title(f'Sensor Correlation Matrix\n'
             f'{high_corr_count//2} pairs with |r| > 0.9 (annotated) → 8 redundant channels dropped',
             fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('/kaggle/working/fig9_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"Saved fig9_correlation_heatmap.png")
print(f"  Found {high_corr_count//2} pairs with |r| > 0.9")

print("\nBoth EDA figures saved to /kaggle/working/ — download from Output tab.")

Saved fig8_flight_length.png
  Flight length — median: 4,402, p95: 9,212, max: 30,059

Sampling sensor data for correlation heatmap...
  Computed correlation on 201,393 sensor rows
Saved fig9_correlation_heatmap.png
  Found 14 pairs with |r| > 0.9

Both EDA figures saved to /kaggle/working/ — download from Output tab.


In [12]:
# SPLIT COMPARISON
#stratified random vs StratifiedGroupKFold

print("SPLIT STRATEGY COMPARISON")
print("=" * 60)

# Stratified Random
from sklearn.model_selection import train_test_split as tts
tr_a, tmp_a = tts(labeled.index, test_size=0.30,
                  stratify=labeled['rul_2d'], random_state=RANDOM_SEED)
va_a, te_a  = tts(tmp_a, test_size=0.50,
                  stratify=labeled.loc[tmp_a,'rul_2d'], random_state=RANDOM_SEED)

print("\nOption A — Stratified Random (current):")
for name, idx in [('train',tr_a),('val',va_a),('test',te_a)]:
    vc = labeled.loc[idx,'rul_2d'].value_counts(normalize=True)
    print(f"  {name:5s}: {len(idx):,} flights | "
          f"class0: {vc.get(0,0)*100:.1f}%  class1: {vc.get(1,0)*100:.1f}%")
print("  Leakage: present by design — ~3,506 events span train/test")
print("           (bounded: avg 5-6 flights per event, marginal model advantage)")

# StratifiedGroupKFold across 4 seeds 
print("\nOption B — StratifiedGroupKFold (event-grouped):")

# Reconstruct event groups
before_f = labeled[labeled['before_after']=='before'].copy()
before_f['event_id'] = before_f.groupby('label')['number_flights_before'].transform(
    lambda x: (x==0).cumsum()
)
before_f['group'] = before_f['label'].astype(str)+'_evt_'+before_f['event_id'].astype(str)
after_f  = labeled[labeled['before_after']=='after'].copy()
after_f['group'] = after_f['label'].astype(str)+'_after'
lab_grp  = pd.concat([before_f, after_f]).sort_index()
print(f"  Reconstructed: {lab_grp['group'].nunique():,} maintenance events")

for seed in [42, 0, 123, 7]:
    sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    tr_rel, tmp_rel = next(sgkf.split(lab_grp, lab_grp['rul_2d'], groups=lab_grp['group']))
    sgkf2 = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)
    va_rel, te_rel = next(sgkf2.split(
        lab_grp.iloc[tmp_rel], lab_grp.iloc[tmp_rel]['rul_2d'],
        groups=lab_grp.iloc[tmp_rel]['group']
    ))
    tr_idx = lab_grp.index[tr_rel]
    va_idx = lab_grp.index[tmp_rel[va_rel]]
    te_idx = lab_grp.index[tmp_rel[te_rel]]

    # Leakage check
    lab_grp['_split_tmp'] = 'train'
    lab_grp.loc[va_idx,'_split_tmp'] = 'val'
    lab_grp.loc[te_idx,'_split_tmp'] = 'test'
    leaked = (lab_grp.groupby('group')['_split_tmp'].nunique() > 1).sum()

    tr_r = lab_grp.loc[tr_idx,'rul_2d'].value_counts(normalize=True).get(0,0)
    va_r = lab_grp.loc[va_idx,'rul_2d'].value_counts(normalize=True).get(0,0)
    te_r = lab_grp.loc[te_idx,'rul_2d'].value_counts(normalize=True).get(0,0)
    te_flag = 'OK' if abs(te_r-tr_r)<0.05 else 'WARNING'

    print(f"\n  seed={seed}: leakage={leaked} | "
          f"train={tr_r*100:.1f}%  val={va_r*100:.1f}%  test={te_r*100:.1f}% [{te_flag}]")

lab_grp.drop(columns=['_split_tmp'], inplace=True)

#Summary 
print("\n" + "=" * 60)
print("CONCLUSION:")
print("  Stratified random : perfect balance (40.4% across all splits)")
print("                      event leakage present (bounded, documented)")
print("  StratifiedGroupKFold: zero leakage verified")
print("                        class balance unstable across seeds")
print("                        (test class0 ranges 23%-58%)")
print("  Decision: stratified random adopted for stable cross-model")
print("  comparison. Leakage documented as limitation per Yang & Desell 2022.")
print("=" * 60)

SPLIT STRATEGY COMPARISON

Option A — Stratified Random (current):
  train: 13,678 flights | class0: 40.4%  class1: 59.6%
  val  : 2,931 flights | class0: 40.4%  class1: 59.6%
  test : 2,931 flights | class0: 40.4%  class1: 59.6%
  Leakage: present by design — ~3,506 events span train/test
           (bounded: avg 5-6 flights per event, marginal model advantage)

Option B — StratifiedGroupKFold (event-grouped):
  Reconstructed: 3,506 maintenance events

  seed=42: leakage=0 | train=40.9%  val=45.2%  test=35.1% [WARNING]

  seed=0: leakage=0 | train=56.0%  val=22.7%  test=29.7% [WARNING]

  seed=123: leakage=0 | train=43.6%  val=25.8%  test=58.3% [WARNING]

  seed=7: leakage=0 | train=45.5%  val=22.6%  test=57.1% [WARNING]

CONCLUSION:
  Stratified random : perfect balance (40.4% across all splits)
                      event leakage present (bounded, documented)
  StratifiedGroupKFold: zero leakage verified
                        class balance unstable across seeds
                   

In [13]:
# Fit RobustScaler on training sample

train_idx_set = set(train_idx)

def sample_train_partition(df):
    df = df[df.index.isin(train_idx_set)]
    if len(df) == 0:
        return pd.DataFrame(columns=KEEP_CHANNELS)
    df = df[[c for c in KEEP_CHANNELS if c in df.columns]].fillna(0)
    return df.sample(n=min(200, len(df)), random_state=42)

print('Fitting RobustScaler on training data sample...')
scaler_sample = flight_data_df.map_partitions(sample_train_partition).compute()
scaler_sample = scaler_sample[[c for c in KEEP_CHANNELS if c in scaler_sample.columns]]

channel_scaler = RobustScaler()
channel_scaler.fit(scaler_sample)

del scaler_sample; gc.collect()

with open(f'{OUT_DIR}/channel_scaler.pkl', 'wb') as f:
    pickle.dump(channel_scaler, f)
print('Scaler fitted. Saved: channel_scaler.pkl')

Fitting RobustScaler on training data sample...
Scaler fitted. Saved: channel_scaler.pkl


In [14]:
# Per-flight feature extraction function

STAT_NAMES = ['mean','std','min','max','range','p05','p25','p50','p75','p95',
              'iqr','slope','first','last','delta','skew','kurt','cv','miss_flag']

def extract_flight_features(group):
    out = {}
    T   = len(group)

    # Per-channel descriptive statistics
    for ch in KEEP_CHANNELS:
        pfx = ch + '__'

        if ch not in group.columns:
            for s in STAT_NAMES: out[pfx+s] = np.nan
            continue

        raw = group[ch].values.astype(np.float64)

        # miss_flag: channel was entirely missing 
        is_missing = (raw.std() < 1e-4) and (abs(raw.mean()) < 1e-4)
        out[pfx+'miss_flag'] = int(is_missing)

        if is_missing or len(raw) < 3:
            for s in STAT_NAMES[:-1]: out[pfx+s] = np.nan
            continue


        out[pfx+'mean']  = raw.mean()
        out[pfx+'std']   = raw.std()
        out[pfx+'min']   = raw.min()
        out[pfx+'max']   = raw.max()
        out[pfx+'range'] = raw.max() - raw.min()


        p05, p25, p50, p75, p95 = np.percentile(raw, [5, 25, 50, 75, 95])
        out[pfx+'p05'] = p05; out[pfx+'p25'] = p25; out[pfx+'p50'] = p50
        out[pfx+'p75'] = p75; out[pfx+'p95'] = p95; out[pfx+'iqr'] = p75 - p25

        x = np.arange(len(raw))
        out[pfx+'slope'] = float(np.polyfit(x, raw, 1)[0])  # linear drift across flight
        out[pfx+'first'] = float(raw[0])    # value at flight start
        out[pfx+'last']  = float(raw[-1])   # value at flight end
        out[pfx+'delta'] = float(raw[-1] - raw[0])  # total drift

        out[pfx+'skew'] = float(sp_skew(raw))
        out[pfx+'kurt'] = float(sp_kurtosis(raw))
        out[pfx+'cv']   = float(raw.std() / (abs(raw.mean()) + 1e-9))

    # Flight phase proportions 
    if 'E1 RPM' in group.columns and 'VSpd' in group.columns:
        rpm  = group['E1 RPM'].values.astype(np.float64)
        vspd = group['VSpd'].values.astype(np.float64)
        flying  = rpm > 1000          
        ground  = ~flying
        climb   = flying & (vspd >  100)   
        descent = flying & (vspd < -100)   
        cruise  = flying & ~climb & ~descent
        out['phase__ground_pct']  = float(ground.mean())
        out['phase__climb_pct']   = float(climb.mean())
        out['phase__cruise_pct']  = float(cruise.mean())
        out['phase__descent_pct'] = float(descent.mean())
    else:
        for k in ['phase__ground_pct','phase__climb_pct',
                  'phase__cruise_pct','phase__descent_pct']: out[k] = np.nan

    # Cross-channel physics features
    def safe_ratio(n_ch, d_ch, d_min=0):
        if n_ch not in group.columns or d_ch not in group.columns: return np.nan
        n = group[n_ch].values.astype(np.float64)
        d = group[d_ch].values.astype(np.float64)
        ok = d > d_min
        return float((n[ok]/(d[ok]+1e-9)).mean()) if ok.sum() > 0 else np.nan

    out['cross__fuel_efficiency']  = safe_ratio('E1 FFlow', 'E1 RPM', d_min=800)
    out['cross__oil_stress']       = safe_ratio('E1 OilT', 'E1 OilP', d_min=10)
    out['cross__cht_egt_ratio']    = safe_ratio('E1 CHT1', 'E1 EGT1', d_min=100)

    if 'FQtyL' in group.columns and 'FQtyR' in group.columns:
        l = group['FQtyL'].values.astype(np.float64)
        r = group['FQtyR'].values.astype(np.float64)
        out['cross__total_fuel']     = float((l+r).mean())   # total fuel quantity
        out['cross__fuel_imbalance'] = float(abs(l-r).mean())  # asymmetric burn
    else:
        out['cross__total_fuel'] = out['cross__fuel_imbalance'] = np.nan

    if 'volt1' in group.columns:
        v = group['volt1'].values.astype(np.float64)
        out['cross__volt_drop'] = float(v.mean() - v.min())
    else:
        out['cross__volt_drop'] = np.nan

    if 'amp1' in group.columns:
        out['cross__elec_load'] = float(group['amp1'].values.astype(np.float64).mean())
    else:
        out['cross__elec_load'] = np.nan

    #Metadata
    out['meta__flight_length'] = T

    return out

# Smoke test
_test = pd.DataFrame(np.random.randn(100, len(KEEP_CHANNELS)), columns=KEEP_CHANNELS)
_f    = extract_flight_features(_test)
print(f'Features per flight  : {len(_f)}')
print(f'  Sensor stats       : {len([k for k in _f if "__" in k and not k.startswith(("phase","cross","meta"))])}')
print(f'  Phase features     : {len([k for k in _f if k.startswith("phase")])}')
print(f'  Cross-channel      : {len([k for k in _f if k.startswith("cross")])}')
print(f'  Metadata           : {len([k for k in _f if k.startswith("meta")])}')
del _test, _f

Features per flight  : 297
  Sensor stats       : 285
  Phase features     : 4
  Cross-channel      : 7
  Metadata           : 1


In [15]:
# Main processing loop

labeled_set  = set(labeled.index)
valid_list   = sorted(labeled.index)
n_flights    = len(valid_list)
flight_pos   = {idx: i for i, idx in enumerate(valid_list)}
n_channels   = len(KEEP_CHANNELS)

seq_path = f'{OUT_DIR}/sequences.npy'
seq_mmap = open_memmap(
    seq_path, mode='w+', dtype=np.float16,
    shape=(n_flights, SEQ_LEN, n_channels)
)

seq_written    = np.zeros(n_flights, dtype=bool)
seq_indices    = np.array(valid_list)
tabular_records = {}

print(f'Flights to process : {n_flights:,}')
print(f'Sequence shape     : ({n_flights}, {SEQ_LEN}, {n_channels})')
print(f'Seq file on disk   : ~{n_flights*SEQ_LEN*n_channels*2/1e9:.1f} GB')
print()

for part_i in tqdm(range(flight_data_df.npartitions), desc='Partitions'):

    # Load one partition
    part = flight_data_df.get_partition(part_i).compute()

    part = part[part.index.isin(labeled_set)]
    if len(part) == 0:
        del part; gc.collect(); continue

    # Sort timesteps
    if 'timestep' in part.columns:
        part = part.sort_values('timestep')

    # Impute per flight 
    sensor_cols = [c for c in ALL_CHANNELS if c in part.columns]
    part[sensor_cols] = (
        part.groupby(level=0)[sensor_cols]
        .transform(lambda x: x.ffill().bfill())
    )
    part[sensor_cols] = part[sensor_cols].fillna(0)  # remaining = full-channel NaN

    # Process each flight
    for idx, group in part.groupby(level=0):
        if idx not in flight_pos: continue
        pos = flight_pos[idx]

        tabular_records[idx] = extract_flight_features(group)

        keep_cols = [c for c in KEEP_CHANNELS if c in group.columns]
        raw_arr   = group[keep_cols].values.astype(np.float32)

        # Handle missing channels
        if len(keep_cols) < n_channels:
            full_arr = np.zeros((len(raw_arr), n_channels), dtype=np.float32)
            for ci, ch in enumerate(KEEP_CHANNELS):
                if ch in keep_cols:
                    full_arr[:, ci] = raw_arr[:, keep_cols.index(ch)]
            raw_arr = full_arr

        scaled = channel_scaler.transform(raw_arr).astype(np.float16)

        T = scaled.shape[0]
        if T >= SEQ_LEN:
            seq_mmap[pos] = scaled[-SEQ_LEN:]
        else:
            seq_mmap[pos, SEQ_LEN-T:, :] = scaled

        seq_written[pos] = True

    del part; gc.collect()

del seq_mmap; gc.collect()

print(f'\nFlights processed  : {seq_written.sum():,} / {n_flights:,}')
if (~seq_written).sum() > 0:
    print(f'WARNING: {(~seq_written).sum()} flights in labels not found in parquet')
print(f'Tabular records    : {len(tabular_records):,}')

Flights to process : 19,540
Sequence shape     : (19540, 9212, 15)
Seq file on disk   : ~5.4 GB



Partitions:   0%|          | 0/401 [00:00<?, ?it/s]


Flights processed  : 19,540 / 19,540
Tabular records    : 19,540


In [16]:
#Assemble tabular features

tabular_df = pd.DataFrame.from_dict(tabular_records, orient='index')
tabular_df.index.name = 'Master Index'

tabular_full = tabular_df.join(
    labeled[['rul_2d','rul_5d','rul_10d','label','label_grouped','split']],
    how='inner'
)

feat_cols = [c for c in tabular_df.columns]
nan_pct   = tabular_full[feat_cols].isna().mean().mean() * 100
print(f'Feature columns  : {len(feat_cols)}')
print(f'Overall NaN rate : {nan_pct:.1f}%')
if nan_pct > 10:
    top_nan = tabular_full[feat_cols].isna().mean().sort_values(ascending=False).head(5)
    print('Top NaN features:'); print(top_nan[top_nan>0])

tabular_full.to_csv(f'{OUT_DIR}/tabular_features.csv')
print(f'\nSaved: tabular_features.csv  {tabular_full.shape}')

train_mask = tabular_full['split'] == 'train'
tabular_scaler = RobustScaler()
tabular_scaler.fit(tabular_full.loc[train_mask, feat_cols].fillna(0))

tabular_scaled = tabular_full.copy()
tabular_scaled[feat_cols] = tabular_scaler.transform(tabular_full[feat_cols].fillna(0))
tabular_scaled.to_csv(f'{OUT_DIR}/tabular_features_scaled.csv')

with open(f'{OUT_DIR}/tabular_scaler.pkl', 'wb') as f:
    pickle.dump(tabular_scaler, f)

print(f'Saved: tabular_features_scaled.csv')
print(f'Saved: tabular_scaler.pkl')

del tabular_df, tabular_records; gc.collect()

Feature columns  : 297
Overall NaN rate : 1.2%

Saved: tabular_features.csv  (19540, 303)
Saved: tabular_features_scaled.csv
Saved: tabular_scaler.pkl


0

In [17]:
# Trim sequence index

seq_indices_final = seq_indices[seq_written]
np.save(f'{OUT_DIR}/seq_indices.npy', seq_indices_final)

seq_v  = np.load(seq_path, mmap_mode='r')
sidx_v = np.load(f'{OUT_DIR}/seq_indices.npy')
print(f'sequences.npy shape : {seq_v.shape}  dtype={seq_v.dtype}')
print(f'seq_indices.npy     : {len(sidx_v):,} entries')
del seq_v
print('Saved: sequences.npy  seq_indices.npy')

sequences.npy shape : (19540, 9212, 15)  dtype=float16
seq_indices.npy     : 19,540 entries
Saved: sequences.npy  seq_indices.npy


In [18]:
#Quality checks

print('='*55)
print('QUALITY CHECKS')
print('='*55)

ldf = pd.read_csv(f'{OUT_DIR}/labels.csv', index_col='Master Index')
print(f'\n[labels.csv]  {ldf.shape}')
print(f'  Splits: {ldf["split"].value_counts().to_dict()}')
for tau in TAU_VALUES:
    vc = ldf[f'rul_{tau}d'].value_counts().sort_index()
    print(f'  rul_{tau}d: class0={vc.get(0,0):,}  class1={vc.get(1,0):,}')
assert ldf['split'].nunique() == 3
assert ldf['rul_2d'].isna().sum() == 0
print('  OK')

tdf  = pd.read_csv(f'{OUT_DIR}/tabular_features.csv', index_col='Master Index')
fcols = [c for c in tdf.columns if '__' in c]
print(f'\n[tabular_features.csv]  {tdf.shape}')
print(f'  Feature columns : {len(fcols)}')
print(f'  NaN rate        : {tdf[fcols].isna().mean().mean()*100:.1f}%')
print(f'  Train rows      : {(tdf["split"]=="train").sum():,}')
assert len(tdf) > 0 and 'rul_2d' in tdf.columns
print('  OK')

seq  = np.load(seq_path, mmap_mode='r')
sidx = np.load(f'{OUT_DIR}/seq_indices.npy')
print(f'\n[sequences.npy]  {seq.shape}  dtype={seq.dtype}')
print(f'  Non-zero rows   : {(seq.reshape(seq.shape[0],-1).sum(axis=1)!=0).sum():,}')
assert seq.shape[0] == len(sidx)
assert seq.shape[2] == len(KEEP_CHANNELS)
print('  OK')

common = len(set(sidx) & set(ldf.index))
print(f'\n[Alignment]  {common:,}/{len(sidx):,} sequences have labels')
assert common > 0.95 * len(sidx)
print('  OK')

del seq, tdf, ldf; gc.collect()
print('\nAll checks passed')

QUALITY CHECKS

[labels.csv]  (19540, 10)
  Splits: {'train': 13678, 'val': 2931, 'test': 2931}
  rul_2d: class0=7,889  class1=11,651
  rul_5d: class0=10,119  class1=9,421
  rul_10d: class0=10,464  class1=9,076
  OK

[tabular_features.csv]  (19540, 303)
  Feature columns : 297
  NaN rate        : 1.2%
  Train rows      : 13,678
  OK

[sequences.npy]  (19540, 9212, 15)  dtype=float16
  Non-zero rows   : 19,540
  OK

[Alignment]  19,540/19,540 sequences have labels
  OK

All checks passed


In [19]:
#Output summary

print('='*55)
print('PIPELINE COMPLETE — files in /kaggle/working/')
print('='*55)
for fname in sorted(os.listdir(OUT_DIR)):
    fsize = os.path.getsize(f'{OUT_DIR}/{fname}')
    unit  = 'GB' if fsize > 1e9 else 'MB'
    size  = fsize/1e9 if fsize > 1e9 else fsize/1e6
    print(f'  {fname:<45}  {size:6.1f} {unit}')

print('''
TEAMMATE GUIDE

EVERYONE: load labels first:
  import pandas as pd, numpy as np
  BASE   = "/kaggle/input/aircraft-pipeline"
  labels = pd.read_csv(f"{BASE}/labels.csv", index_col="Master Index")

SEE LIN (XGBoost):
  tab   = pd.read_csv(f"{BASE}/tabular_features.csv", index_col="Master Index")
  feats = [c for c in tab.columns if "__" in c]
  X_train = tab[tab["split"]=="train"][feats].fillna(0)
  y_train = tab[tab["split"]=="train"]["rul_2d"].values
  X_val   = tab[tab["split"]=="val"][feats].fillna(0)
  y_val   = tab[tab["split"]=="val"]["rul_2d"].values
  X_test  = tab[tab["split"]=="test"][feats].fillna(0)
  y_test  = tab[tab["split"]=="test"]["rul_2d"].values
  # Stretch goal: swap rul_2d for label_grouped (10-class + other)

TANYA / LINH / HELEN (TS models):
  X   = np.load(f"{BASE}/sequences.npy", mmap_mode="r")  # (N, 9212, 15)
  idx = np.load(f"{BASE}/seq_indices.npy")
  y     = labels.loc[idx, "rul_2d"].values   # swap for rul_5d / rul_10d
  split = labels.loc[idx, "split"].values
  X_train, y_train = X[split=="train"], y[split=="train"]
  X_val,   y_val   = X[split=="val"],   y[split=="val"]
  X_test,  y_test  = X[split=="test"],  y[split=="test"]

CHANNEL ORDER in sequences (axis=2):
  0:volt1  1:amp1  2:FQtyL  3:FQtyR  4:E1 FFlow
  5:E1 OilT  6:E1 OilP  7:E1 RPM  8:E1 CHT1  9:E1 EGT1
  10:OAT  11:IAS  12:VSpd  13:NormAc  14:AltMSL
  Already: RobustScaled (train-fitted), left-padded 0, float16

FOR SENSITIVITY ANALYSIS (Linh):
  y = labels.loc[idx, "rul_2d"].values   # tau=2 (paper baseline)
  y = labels.loc[idx, "rul_5d"].values   # tau=5
  y = labels.loc[idx, "rul_10d"].values  # tau=10

LIMITATIONS:
  - Group-aware split not implemented (flights from same maintenance
    event may span train/test). Flag in report as future work.
  - Flights spanning multiple parquet partitions: rare edge case,
    handled by fillna(0) at partition boundary.
''')

PIPELINE COMPLETE — files in /kaggle/working/
  __notebook__.ipynb                                0.1 MB
  channel_scaler.pkl                                0.0 MB
  fig1_stratified_random.png                        0.1 MB
  fig2_grouped_unstable.png                         0.1 MB
  fig3_decision_summary.png                         0.1 MB
  fig4_class_balance.png                            0.1 MB
  fig5_t1_vs_tp1.png                                0.2 MB
  fig6_operating_ranges.png                         0.2 MB
  fig7_tp1_vs_tp2.png                               0.2 MB
  fig8_flight_length.png                            0.1 MB
  fig9_correlation_heatmap.png                      0.2 MB
  labels.csv                                        1.7 MB
  seq_indices.npy                                   0.2 MB
  sequences.npy                                     5.4 GB
  tabular_features.csv                             68.3 MB
  tabular_features_scaled.csv                     105.7 MB
  tabular_